# Lesson 6

## Video 1 — Context Engineering

In [1]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Import the Anthropic SDK
import anthropic

# Create the client
client = anthropic.Anthropic()

# Model used in this lesson
MODEL = "claude-sonnet-5-5"

### Relevant context vs noise

In [26]:
# The task only needs a few relevant facts
question = "Why can the customer not access analytics?"

relevant_context = """
The customer is on the Standard plan.
Only the Premium plan includes access to the analytics dashboard.
"""

In [27]:
# A focused prompt contains only what the task needs
focused_prompt = f"""
Answer the question using only the context below.

<context>
{relevant_context}
</context>

<question>
{question}
</question>

Return a concise answer.
"""

resp = client.messages.create(
    model=MODEL,
    max_tokens=120,
    messages=[
        {"role": "user", "content": focused_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

Claude: The customer is on the Standard plan, and only the Premium plan includes access to the analytics dashboard.


In [28]:
# Extra information can add noise without helping the task
noisy_context = """
The customer joined in 2024.
They use dark mode.
Their team has 14 members.
They opened two support tickets last month.
The customer is on the Standard plan.
Only the Premium plan includes access to the analytics dashboard.
Their office is in London.
They usually log in from Chrome.
"""

noisy_prompt = f"""
Answer the question using only the context below.

<context>
{noisy_context}
</context>

<question>
{question}
</question>

Return a concise answer.
"""

resp = client.messages.create(
    model=MODEL,
    max_tokens=120,
    messages=[
        {"role": "user", "content": noisy_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

Claude: The customer is on the Standard plan, and only the Premium plan includes access to the analytics dashboard.


## Video 2 — The Context Window and Long Conversations

In [29]:
# Start with a small conversation history
messages = [
    {"role": "user", "content": "My project is called Claude Assist."},
    {"role": "assistant", "content": "Got it."},
]

print("Messages:", len(messages))
messages

Messages: 2


[{'role': 'user', 'content': 'My project is called Claude Assist.'},
 {'role': 'assistant', 'content': 'Got it.'}]

In [30]:
# Add more turns
messages.extend([
    {"role": "user", "content": "It uses Python."},
    {"role": "assistant", "content": "Understood."},
    {"role": "user", "content": "We are adding structured output."},
    {"role": "assistant", "content": "That makes sense."},
])

print("Messages:", len(messages))
messages

Messages: 6


[{'role': 'user', 'content': 'My project is called Claude Assist.'},
 {'role': 'assistant', 'content': 'Got it.'},
 {'role': 'user', 'content': 'It uses Python.'},
 {'role': 'assistant', 'content': 'Understood.'},
 {'role': 'user', 'content': 'We are adding structured output.'},
 {'role': 'assistant', 'content': 'That makes sense.'}]

In [31]:
# Each new API call can resend the entire messages list
# More turns = more context sent again on the next request

print(f"The next request would resend {len(messages)} messages.")

The next request would resend 6 messages.


## Video 3 — Requesting Structured JSON

In [2]:
from prompts import extract_fields_prompt

In [3]:
# Sample text to extract fields from
text = """
Anas is preparing a course lesson about context engineering and structured output.
"""

# Build the prompt from prompts.py
json_prompt = extract_fields_prompt(text)

# Inspect exactly what will be sent
print(json_prompt)

Extract the key information from the text below.

Return ONLY valid JSON with exactly these keys:
- "name"
- "topic"

<text>

Anas is preparing a course lesson about context engineering and structured output.

</text>

Do not include markdown, code fences, or any text outside the JSON object.


In [4]:
# Ask Claude to return only the named JSON fields
resp = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role": "user", "content": json_prompt}
    ]
)

# Claude's reply still arrives as text
raw = "".join(block.text for block in resp.content if block.type == "text")
print(raw)

{"name": "Anas", "topic": "context engineering and structured output"}


In [5]:
print(type(raw))

<class 'str'>


## Video 4 — Parsing JSON Safely

In [6]:
raw

'{"name": "Anas", "topic": "context engineering and structured output"}'

In [7]:
# Python's built-in json library parses JSON text
import json

data = json.loads(raw)

print(data)
print(type(data))

{'name': 'Anas', 'topic': 'context engineering and structured output'}
<class 'dict'>


In [8]:
# Access individual fields like a normal Python dictionary
print("Name:", data["name"])
print("Topic:", data["topic"])

Name: Anas
Topic: context engineering and structured output


### Handle invalid JSON

In [9]:
from output import parse_json

In [10]:
# Valid JSON returns a Python dictionary
valid_raw = '{"name": "Anas", "topic": "AI engineering"}'

parse_json(valid_raw)

{'name': 'Anas', 'topic': 'AI engineering'}

In [11]:
# Invalid JSON returns an error object instead of crashing
invalid_raw = "Here is the result: Anas is working on AI engineering."

parse_json(invalid_raw)

{'error': 'invalid JSON',
 'raw': 'Here is the result: Anas is working on AI engineering.'}

### Use the Full Structured-Output Helper

In [12]:
# output.py can build the prompt, call Claude and parse the reply
from output import extract_fields

result = extract_fields(
    client=client,
    model=MODEL,
    text="Maya is writing documentation about API security."
)

result

{'name': 'Maya', 'topic': 'API security'}

### Keep Conversation History in Bounds

In [13]:
# Import the history helper
from output import trim_history

In [14]:
# Build a longer sample history
messages = [
    {"role": "user", "content": f"User message {i}"}
    if i % 2 == 0
    else {"role": "assistant", "content": f"Assistant reply {i}"}
    for i in range(14)
]

print("Before:", len(messages))

Before: 14


In [15]:
# Keep only the most recent 6 messages
trimmed_messages = trim_history(messages, n=6)

print("After:", len(trimmed_messages))
trimmed_messages

After: 6


[{'role': 'user', 'content': 'User message 8'},
 {'role': 'assistant', 'content': 'Assistant reply 9'},
 {'role': 'user', 'content': 'User message 10'},
 {'role': 'assistant', 'content': 'Assistant reply 11'},
 {'role': 'user', 'content': 'User message 12'},
 {'role': 'assistant', 'content': 'Assistant reply 13'}]

In [16]:
# In a real conversation loop, trim before sending
messages = trim_history(messages, n=10)

print(f"Sending {len(messages)} recent messages to Claude.")

Sending 10 recent messages to Claude.


### Final Validation

In [17]:
# 1. Valid JSON should become a dictionary
print(parse_json('{"name": "Sam", "topic": "Python"}'))

# 2. Invalid JSON should return an error object
print(parse_json("not valid json"))

# 3. History should keep only the most recent messages
print(trim_history(list(range(20)), n=5))

{'name': 'Sam', 'topic': 'Python'}
{'error': 'invalid JSON', 'raw': 'not valid json'}
[15, 16, 17, 18, 19]
